# 02 — Feature Engineering
The raw data encodes numeric quantities as ordered text bins. `src/features.py`
decodes them to numeric midpoints and adds **investigator-style derived
features**. Everything statistical is learned inside the sklearn pipeline
(`fit` on training folds only) — no leakage, identical behavior in the API.

**Missing-value strategy (explicit, per column):**
- `Age` — 0 is the dataset's missing marker (319 rows after cleaning) → flag
  column `age_was_missing`, impute with the **training** median.
- All other columns are complete after dropping the single corrupt row.

In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))
import pandas as pd
from src.preprocessing import PROCESSED_DIR, TARGET
from src.features import FraudFeatureEngineer, ORDINAL_MAPS

train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
X, y = train.drop(columns=[TARGET]), train[TARGET]
print("train:", X.shape)

train: (12335, 31)


## Ordinal decoding map (documented, deterministic)

In [2]:
rows = [(col, k, v) for col, m in ORDINAL_MAPS.items() for k, v in m.items()]
pd.DataFrame(rows, columns=["column", "bin label", "numeric value"]).head(20)

,column,bin label,numeric value
0,VehiclePrice,less than 20000,15000.0
1,VehiclePrice,20000 to 29000,24500.0
2,VehiclePrice,30000 to 39000,34500.0
3,VehiclePrice,40000 to 59000,49500.0
4,VehiclePrice,60000 to 69000,64500.0
5,VehiclePrice,more than 69000,80000.0
6,Days_Policy_Accident,none,0.0
7,Days_Policy_Accident,1 to 7,4.0
8,Days_Policy_Accident,8 to 15,11.5
9,Days_Policy_Accident,15 to 30,22.5


## Fit + transform: what the model actually sees

In [3]:
fe = FraudFeatureEngineer().fit(X, y)
Xt = fe.transform(X)
print("engineered shape:", Xt.shape)
new_cols = ["price_per_vehicle_year", "price_z_in_category", "short_gap_flag",
            "recent_address_change_flag", "claim_month_mismatch",
            "no_report_no_witness", "make_freq", "age_was_missing"]
Xt[new_cols].describe().T.round(3)

engineered shape: (12335, 38)


,count,mean,std,min,25%,50%,75%,max
price_per_vehicle_year,12335.0,6194.137,11949.844,1500.000,3062.500,3500.000,4900.000,80000.000
price_z_in_category,12335.0,-0.000,1.000,-4.357,-0.413,-0.413,0.178,2.866
short_gap_flag,12335.0,0.004,0.066,0.000,0.000,0.000,0.000,1.000
recent_address_change_flag,12335.0,0.012,0.110,0.000,0.000,0.000,0.000,1.000
claim_month_mismatch,12335.0,0.255,0.436,0.000,0.000,0.000,1.000,1.000
no_report_no_witness,12335.0,0.969,0.173,0.000,1.000,1.000,1.000,1.000
make_freq,12335.0,0.174,0.068,0.000,0.150,0.183,0.251,0.251
age_was_missing,12335.0,0.021,0.143,0.000,0.000,0.000,0.000,1.000


## Do the derived red flags actually separate fraud?

In [4]:
check = Xt[new_cols[:6]].copy()
check["fraud"] = y.values
print((check.groupby("fraud").mean().T * 100).round(1).rename(
    columns={0: "clean %", 1: "fraud %"}))

fraud                        clean %   fraud %
price_per_vehicle_year      611432.7  744827.9
price_z_in_category             -1.1      17.4
short_gap_flag                   0.4       1.1
recent_address_change_flag       1.2       1.6
claim_month_mismatch            25.1      31.4
no_report_no_witness            96.8      98.2


The processed train/test parquet files are produced by
`python -m src.preprocessing`; models consume them directly. The engineered
features live inside the model pipeline itself, so the API needs only raw
claim fields.